# P3 — CNN inicial

## Objetivo científico
Establecer una línea base entrenada desde cero.

La comparación posterior con ResNet50 permitirá responder:

> ¿Cuánto mejora el desempeño al utilizar Transfer Learning?

In [1]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## 1. Detectar dispositivo

In [2]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo:", device)
print("CUDA:", torch.cuda.is_available())

Dispositivo: cpu
CUDA: False


## 2. Construir DataLoaders

In [3]:
from src.data import make_loaders

df, loaders = make_loaders(
    ROOT / cfg["data"]["supervised_split"],
    root=ROOT,
    input_size=cfg["model"]["input_size"],
    batch_size=cfg["training"]["batch_size"],
    num_workers=cfg["training"]["num_workers"]
)

for key, loader in loaders.items():
    print(key, len(loader.dataset))

train 140
val 30
test_internal 30


## 3. Comprobar un batch real

In [4]:
images, labels, paths = next(
    iter(loaders["train"])
)

print("Batch images:", images.shape)
print("Batch labels:", labels.shape)
print("Primera imagen:", paths[0])

Batch images: torch.Size([16, 3, 224, 224])
Batch labels: torch.Size([16])
Primera imagen: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL\pictures\normal\00084.jpg


## 4. Construir CNN baseline

In [5]:
from src.models import (
    BaselineCNN,
    count_parameters
)

model = BaselineCNN(
    dropout=cfg["model"]["dropout"]
).to(device)

total, trainable = count_parameters(model)

print(model)
print("Parámetros totales:", total)
print("Parámetros entrenables:", trainable)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout

## 5. Validar forward propagation

In [6]:
with torch.no_grad():
    logits = model(
        images.to(device)
    )

print("Input:", images.shape)
print("Output:", logits.shape)

Input: torch.Size([16, 3, 224, 224])
Output: torch.Size([16])


## 6. Entrenamiento con Early Stopping

In [ ]:
from src.training import train_model
import pandas as pd

checkpoint = ROOT/"models/baseline_best.pt"
history_path = ROOT/"results/training/baseline_history.csv"
history_path.parent.mkdir(parents=True, exist_ok=True)

if checkpoint.exists() and history_path.exists():
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    history = pd.read_csv(history_path)
    print("Se reutiliza baseline_best.pt y su historial reproducible.")
else:
    model, history = train_model(
        model,
        loaders["train"],
        loaders["val"],
        epochs=20,
        lr=1e-3,
        weight_decay=1e-4,
        patience=5,
        checkpoint=checkpoint,
        device=device
    )
    history.to_csv(history_path, index=False)

display(history.tail())


## 7. Curvas de aprendizaje

In [ ]:
import matplotlib.pyplot as plt
OUT = ROOT/"results/training"; OUT.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(history["epoch"], history["train_loss"], label="Train")
ax.plot(history["epoch"], history["val_loss"], label="Validation")
ax.set_title("CNN baseline — Loss"); ax.set_xlabel("Época"); ax.set_ylabel("Binary Cross Entropy")
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(OUT/"baseline_loss.png", dpi=220, bbox_inches="tight"); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(history["epoch"], history["train_accuracy"], label="Train")
ax.plot(history["epoch"], history["val_accuracy"], label="Validation")
ax.set_title("CNN baseline — Accuracy"); ax.set_xlabel("Época"); ax.set_ylabel("Accuracy")
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(OUT/"baseline_accuracy.png", dpi=220, bbox_inches="tight"); plt.show(); plt.close(fig)
print('P3 exportó historial y curvas de la CNN baseline.')
